# 02 — Soft beam attribution

In scanning / PF-HEDM the indexer must decide which observed spots a
candidate voxel "owns", based on how far the voxel's projected
position sits from the illuminating beam. The legacy behavior is a
**hard window**: a spot counts iff it falls within `ScanPosTol` µm.

`midas-index` generalizes this into **soft attribution**: each
candidate match gets a *weight* in `[0, 1]` from a beam-profile
function. The C backend writes these weights to the
`IndexBest_weights_all.bin` sidecar (1.0 everywhere in the legacy
`none` mode, so downstream code can always rely on the file). This
notebook demonstrates the weighting kernels and how they flow into
`compare_spots` scoring — all on CPU, in a couple of seconds.


In [1]:
import os
# torch + numba (pulled in by the matcher) can both ship a libomp on
# macOS; allow the duplicate so the kernel doesn't abort. Matches the
# package's own test harness.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import math
import torch

from midas_index.compute.soft_attribution import (
    hard_window_fn, soft_top_hat_fn, soft_gaussian_fn,
)

print("torch:", torch.__version__)


torch: 2.11.0


## 1. The three attribution kernels

Each kernel maps a beam-distance `d` (µm) to a weight in `[0, 1]`:

- `hard_window_fn(tol)` — the legacy binary filter (`1` if `d < tol`).
- `soft_top_hat_fn(width, fall_off)` — flat top, linear edge ramp.
- `soft_gaussian_fn(fwhm)` — Gaussian; weight = 0.5 at `d = FWHM/2`.

Configured in `paramstest.txt` via `SoftAttrMode` /
`SoftAttrFwhm` / `SoftAttrTruncate` / `SoftAttrFalloff`.


In [2]:
d = torch.linspace(0, 12, 13, dtype=torch.float64)

hard = hard_window_fn(5.0)(d)
tophat = soft_top_hat_fn(10.0, fall_off_um=4.0)(d)     # half-width 5, ramp to 9
gauss = soft_gaussian_fn(10.0)(d)                       # 0.5 at d = 5

print(f"{'d(µm)':>6} {'hard(<5)':>9} {'top_hat':>9} {'gaussian':>9}")
for i in range(len(d)):
    print(f"{float(d[i]):6.1f} {float(hard[i]):9.3f} "
          f"{float(tophat[i]):9.3f} {float(gauss[i]):9.3f}")

# Sanity checks matching the package's tests.
assert math.isclose(float(soft_gaussian_fn(10.0)(torch.tensor([5.0]))[0]), 0.5, abs_tol=1e-9)
assert math.isclose(float(soft_top_hat_fn(10.0, fall_off_um=4.0)(torch.tensor([6.0]))[0]), 0.75, rel_tol=1e-9)
print("\nGaussian half-max at FWHM/2 ✓   top-hat ramp value ✓")


 d(µm)  hard(<5)   top_hat  gaussian
   0.0     1.000     1.000     1.000
   1.0     1.000     1.000     0.973
   2.0     1.000     1.000     0.895
   3.0     1.000     1.000     0.779
   4.0     1.000     1.000     0.642
   5.0     0.000     1.000     0.500
   6.0     0.000     0.750     0.369
   7.0     0.000     0.500     0.257
   8.0     0.000     0.250     0.170
   9.0     0.000     0.000     0.106
  10.0     0.000     0.000     0.062
  11.0     0.000     0.000     0.035
  12.0     0.000     0.000     0.018

Gaussian half-max at FWHM/2 ✓   top-hat ramp value ✓


## 2. Differentiable in the distance

The soft kernels are smooth, so the per-match weight carries an
autograd gradient w.r.t. the voxel↔beam distance — useful for joint
calibration / grain-mapping. (The hard window is, by design, not
differentiable.)


In [3]:
dd = torch.tensor([1.0, 3.0, 5.0], dtype=torch.float64, requires_grad=True)
w = soft_gaussian_fn(10.0)(dd).sum()
w.backward()
print("d/d(distance) of total gaussian weight:", dd.grad.tolist())
assert torch.autograd.gradcheck(
    lambda x: soft_gaussian_fn(10.0)(x).sum(),
    (torch.tensor([1.0, 3.0, 5.0], dtype=torch.float64, requires_grad=True),),
    eps=1e-6, atol=1e-7,
)
print("gradcheck passed ✓")


d/d(distance) of total gaussian weight: [-0.0539354427565206, -0.1296181755801241, -0.13862943611198905]
gradcheck passed ✓


## 3. Weights feeding the matcher

`compute.matching.compare_spots` accepts a `soft_beam_weight_fn`. When
provided, it returns `weighted_n_matches` / `weighted_frac_matches`
alongside the integer counts (the values that populate
`IndexBest_weights_all.bin`). We build a minimal single-voxel,
single-theoretical-spot scan setup and compare hard vs soft scoring.


In [4]:
from midas_index.compute.matching import (
    compare_spots, build_eta_margins, build_ome_margins,
)

def matching_kwargs():
    return dict(
        eta_margins=build_eta_margins(
            ring_radii={1: 30000.0}, margin_eta=20.0, stepsize_orient_deg=0.5,
            device=torch.device("cpu"), dtype=torch.float64),
        ome_margins=build_ome_margins(
            margin_ome=10.0, stepsize_orient_deg=0.5,
            device=torch.device("cpu"), dtype=torch.float64),
        eta_bin_size=0.1, ome_bin_size=5.0,
        n_eta_bins=3600, n_ome_bins=72,
        rings_to_reject=torch.tensor([], dtype=torch.int64),
        margin_rad=50.0, margin_radial=50.0,
    )

def build_bin(eta, ome, ring_nr, eta_bin, ome_bin, n_eta, n_ome, n_rows):
    pos = ((ring_nr - 1) * n_eta * n_ome
           + int((180 + eta) / eta_bin) * n_ome
           + int((180 + ome) / ome_bin))
    ndata = torch.zeros(2 * (pos + 10), dtype=torch.int32)
    ndata[2 * pos] = n_rows
    return torch.arange(n_rows, dtype=torch.int32), ndata

# One observed spot at ω=90°; voxel at (10, 0) µm → beam projection = 10 µm.
omega = 90.0
obs = torch.tensor(
    [(10.0, 5.0, omega, 30000.0, 101, 1, 12.0, 1.5, 0.0, 0)],
    dtype=torch.float64)
theor = torch.tensor(
    [[0, 0, 0, 0, 0, 0, omega, 0, 0, 1, 10.0, 5.0, 12.0, 0.0]],
    dtype=torch.float64).unsqueeze(0)
valid = torch.ones((1, 1), dtype=torch.bool)
kw = matching_kwargs()
bin_data, bin_ndata = build_bin(12.0, omega, 1, kw["eta_bin_size"],
                                kw["ome_bin_size"], kw["n_eta_bins"],
                                kw["n_ome_bins"], n_rows=1)

common = dict(
    obs=obs, theor=theor, valid=valid,
    bin_data=bin_data, bin_ndata=bin_ndata,
    ref_rad=torch.tensor([30000.0], dtype=torch.float64),
    scan_positions=torch.tensor([4.0], dtype=torch.float64),  # beam at 4 µm
    voxel_xy=torch.tensor([[10.0, 0.0]], dtype=torch.float64),
    scan_pos_tol_um=20.0, **kw,
)

res_hard = compare_spots(**common)
res_soft = compare_spots(**common, soft_beam_weight_fn=soft_top_hat_fn(10.0, fall_off_um=4.0))

print("beam distance = |10 - 4| = 6 µm")
print("hard counts   : n_matches =", int(res_hard.n_matches.item()),
      "| weighted_n_matches =", res_hard.weighted_n_matches)
print("soft (top-hat): n_matches =", int(res_soft.n_matches.item()),
      "| weighted_n_matches =", float(res_soft.weighted_n_matches[0]))
print()
print("→ both count the spot, but soft attribution down-weights it to")
print("  (5 + 4 - 6) / 4 = 0.75 because it sits on the beam-edge ramp.")


beam distance = |10 - 4| = 6 µm
hard counts   : n_matches = 1 | weighted_n_matches = None
soft (top-hat): n_matches = 1 | weighted_n_matches = 0.75

→ both count the spot, but soft attribution down-weights it to
  (5 + 4 - 6) / 4 = 0.75 because it sits on the beam-edge ramp.


## Recap

- Soft attribution turns the hard `ScanPosTol` window into a weighted
  one; the weights land in `IndexBest_weights_all.bin`.
- Three kernels: `hard_window_fn`, `soft_top_hat_fn`, `soft_gaussian_fn`.
- They are autograd-differentiable in beam distance.
- Enable in production via `SoftAttrMode gaussian` (etc.) in
  `paramstest.txt`; mode `none` preserves legacy behavior bit-for-bit.
